# 06 · Equilibration and production MD — PSMA / GCPII (Q04609)

**Milestone 7a.** Runs the membrane system that notebook 05 built and checked (MDReadiness **READY**) in OpenMM, on the Colab GPU.

| Phase | What happens | Simulated time |
|---|---|---|
| **Equilibration** | six steps, CHARMM-GUI style: heat at constant volume, then constant pressure (membrane barostat), releasing the restraints on the protein, the Zn site, glycans and lipid heads step by step | 1.875 ns |
| **Production** | no restraints, constant pressure, 310 K, 4 fs time step (hydrogen mass repartitioning) | 100 ns (`md.production.ns`) |

**Starting point.** `outputs/md_setup/system/system.xml` from notebook 05. It contains two corrections that are *not* in the prmtop, so this notebook never rebuilds the system from the prmtop:
- the hydroxide hydrogen's small LJ core;
- the Zn–ligand bond-length (r0) correction, which keeps the zinc site at the crystal geometry.

**Resumable.** The state is saved to Drive after every segment (25 ps in equilibration, 1 ns in production). If the session ends or your units run out, re-run the notebook: it continues from the last saved segment. Set `MAX_MINUTES` so a run stops cleanly before the session would end.

**Cost.** The system is large (~534,000 atoms). After the first segments, section D prints the measured speed (ns/day) and the GPU hours left. Check that before committing units to production.

**Outputs** (`outputs/md_run/` on Drive):
- `log.csv`: one row per segment (temperature, box, area per lipid, Zn–ligand distances, Cα RMSD, speed);
- `solute.pdb` + `traj/*.dcd`: frames of the protein, Zn site and glycans (every 5 ps in equilibration, 10 ps in production);
- `state.xml`: the latest full state; `equilibrated.pdb`: the full system after equilibration.

## 0 · Setup

In [ ]:
#@title 0a · the pipeline (Colab's own Python: display only)
import os, sys, subprocess, shutil, time
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
DATA_DIR = None     # your own machine: the folder holding 'outputs' (see docs/LOCAL_GPU.md); None in Colab
REPO = "https://github.com/victorolatunde23-dev/psma_structural_modelling.git"
if IN_COLAB and not Path("psma_structural_modelling").exists() and not Path("src/psma_pipeline").exists():
    subprocess.run(["git", "clone", "-q", REPO], check=True)
if Path("psma_structural_modelling").exists():
    os.chdir("psma_structural_modelling")
if IN_COLAB:
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
sys.path.insert(0, str(Path("src").resolve()))
for _m in [m for m in list(sys.modules) if m == "psma_pipeline" or m.startswith("psma_pipeline.")]:
    del sys.modules[_m]
import psma_pipeline
print("psma_pipeline", psma_pipeline.__version__, "| Python", sys.version.split()[0])
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True) \
    if shutil.which("nvidia-smi") else None
print("GPU:", r.stdout.strip() if r and r.returncode == 0 else "NONE - choose Runtime > Change runtime type > GPU (MD on CPU is far too slow)")

In [ ]:
#@title 0b · MD environment (OpenMM; ~5–10 min once per Colab session)
from psma_pipeline import md_env
t0 = time.time()
info = md_env.install_env(Path(".").resolve(), gpu=True)
print(f"ready in {(time.time() - t0) / 60:.1f} min")
for k, v in info.items():
    print(f"  {k}: {v}")
assert md_env.env_ready(), f"MD environment incomplete: see {info.get('install log')}"

In [ ]:
#@title 1 · configuration and the notebook 05 result
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from psma_pipeline import load_config, setup_logging
from psma_pipeline.colab import local_overrides, storage_overrides
setup_logging()
cfg = load_config(overrides=(storage_overrides(True) if IN_COLAB else local_overrides(DATA_DIR)) or None)
OUT = cfg.outputs_dir
sysdir = OUT / "md_setup" / "system"
for f in ("system.xml", "system.prmtop", "system_minimised.pdb", "zn_r0_correction.json"):
    print(f"{f:28s}", "found" if (sysdir / f).exists() else "MISSING")
res05 = OUT / "md_setup" / "system_results.json"
ready = False
if res05.exists():
    r05 = md_env.load_results(res05)
    mdr = r05["tables"].get("MDReadiness")
    if mdr is not None:
        display(mdr)
        ready = mdr.iloc[-1]["Status"] == "PASS"
assert ready, "notebook 05 has not reached MDReadiness READY: finish it first"
eq = cfg.get("md.equilibration")
display(pd.DataFrame(eq["stages"]).rename(columns={"bb": "backbone k", "sc": "side chain / site k", "lip": "lipid head z k"}))
print("production:", cfg.get("md.production"))

## A · Unit tests (MD helpers)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_md_run_offline.py"], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-800:])
assert r.returncode == 0

## B · How long to run in this session

`MAX_MINUTES` is the longest this session should run the MD before stopping cleanly (the state is saved after every segment anyway). Colab Pro sessions usually last up to ~12–24 h; on pay-as-you-go units, choose what you are happy to spend: an L4 uses roughly 1.7–2 units per hour, a T4 about 1.2.

In [ ]:
MAX_MINUTES = 180   #@param {type:"number"}
ARGS = ["--max-minutes", str(MAX_MINUTES)]

## C · Equilibration

Each row of the table below is one 25 ps segment. What to look for:
- **Temperature** close to 310 K;
- **box x/y** shrinking a little during the NPT steps, then levelling off (the bilayer finding its area);
- **area per lipid** settling (POPC with 30 % cholesterol: typically ~45–55 Å², counted over PC + cholesterol);
- **Zn–ligand deviation** staying below 0.3 Å;
- **Cα RMSD** rising slowly as the restraints are released (1–2 Å is normal).

In [ ]:
eqres = md_env.run_stage("equilibrate", OUT, extra=ARGS)
T = eqres["tables"]
for k in ("MDProgress", "MDInfo"):
    if k in T: display(T[k])
if "MDLog" in T and len(T["MDLog"]):
    log06 = T["MDLog"]
    display(log06.tail(12))
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.2))
    t = log06["Time (ps)"] / 1000
    ax[0].plot(t, log06["Temperature (K)"]); ax[0].set_title("temperature (K)")
    ax[1].plot(t, log06["Area per lipid (Å²)"]); ax[1].set_title("area per lipid (Å²)")
    ax[2].plot(t, log06["Max Zn–ligand deviation (Å)"]); ax[2].axhline(0.3, ls="--", c="r"); ax[2].set_title("max Zn–ligand deviation (Å)")
    ax[3].plot(t, log06["Cα RMSD from start (Å)"]); ax[3].set_title("Cα RMSD (Å)")
    for a in ax: a.set_xlabel("time (ns)")
    plt.tight_layout(); plt.show()
if "EquilibrationChecks" in T:
    display(T["EquilibrationChecks"])
status = eqres["extra"].get("status")
print("equilibration:", status)
assert eqres["returncode"] == 0, "equilibration problem: see the output above and Issues"
if status != "complete":
    print("Not finished yet: run this notebook again (Run all) to continue from the last saved segment.")

## D · Production

Starts only after equilibration has finished and passed its checks. Each segment is 1 ns. **Check the estimated GPU hours** in MDProgress after the first segment.

In [ ]:
RUN_PRODUCTION = False   #@param {type:"boolean"}
if RUN_PRODUCTION and eqres["extra"].get("status") == "complete":
    pr = md_env.run_stage("produce", OUT, extra=ARGS)
    P = pr["tables"]
    if "MDProgress" in P: display(P["MDProgress"])
    if "MDLog" in P and len(P["MDLog"]):
        lg = P["MDLog"][P["MDLog"]["Phase"] == "prod"]
        display(lg.tail(10))
        fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
        t = lg["Time (ps)"] / 1000
        ax[0].plot(t, lg["Area per lipid (Å²)"]); ax[0].set_title("area per lipid (Å²)")
        ax[1].plot(t, lg["Max Zn–ligand deviation (Å)"]); ax[1].set_title("max Zn–ligand deviation (Å)")
        ax[2].plot(t, lg["Cα RMSD from start (Å)"]); ax[2].set_title("Cα RMSD (Å)")
        for a in ax: a.set_xlabel("time (ns)")
        plt.tight_layout(); plt.show()
    print("production:", pr["extra"].get("status"))
    assert pr["returncode"] == 0
else:
    print("Production not started: set RUN_PRODUCTION = True once equilibration is complete and you have checked the speed.")